# Phase 4.2: Facility Manager Agent (Subscriber + Publisher)

This notebook subscribes to spectator and movement topics, then publishes queue-state and task-state updates for Section A4.

In [ ]:
# Cell 2 purpose: Import modules, facility-manager logic, and topic helpers for Phase 4.2.
import json
import sys
import time
from pathlib import Path

sys.path.insert(0, str(Path('../src').resolve()))

from simulated_city import mqtt
from simulated_city.config import load_config
from simulated_city.facility_manager import (
    FacilityManagerState,
    process_movement_snapshot,
    process_spectator_event,
 )
from simulated_city.topic_schema import (
    topic_movement_state,
    topic_queue_state,
    topic_spectator_events,
    topic_task_state,
 )

In [ ]:
# Cell 3 purpose: Load config, initialize state, and connect MQTT client for dual subscribe/publish.
app_config = load_config()
if app_config.halftime is None:
    raise ValueError('Missing halftime section in config.yaml')

state = FacilityManagerState(shared_urinal_total=app_config.halftime.capacity.shared_urinal_total)

spectator_topic = topic_spectator_events()
movement_topic = topic_movement_state()
queue_state_topic = topic_queue_state()
task_state_topic = topic_task_state()
mqtt_client = mqtt.connect_mqtt(app_config.mqtt, client_id_suffix='facility-manager')

print(f'Connected to MQTT broker at {app_config.mqtt.host}:{app_config.mqtt.port}')
print(f'Subscribed topics: {spectator_topic}, {movement_topic}')
print(f'Publish topics: {queue_state_topic}, {task_state_topic}')

Connected to MQTT broker at 127.0.0.1:1883
Subscribed topic: stadium/a4/halftime/events/spectator
Publish topic: stadium/a4/halftime/state/queues


In [ ]:
# Cell 4 purpose: Subscribe to spectator/movement events and publish queue/task updates.
received_spectator_events = []
received_movement_events = []
published_queue_states = []
published_task_states = []

def _on_message(client, userdata, msg):
    _ = userdata
    try:
        payload = json.loads(msg.payload.decode('utf-8'))
    except json.JSONDecodeError:
        return

    if msg.topic == spectator_topic:
        received_spectator_events.append(payload)
        queue_state_payload = process_spectator_event(state, payload)
        if queue_state_payload is None:
            return

        ok = mqtt.publish_json_checked(client, queue_state_topic, queue_state_payload, qos=1)
        if ok:
            published_queue_states.append(queue_state_payload)
        return

    if msg.topic == movement_topic:
        received_movement_events.append(payload)
        task_payloads = process_movement_snapshot(state, payload)
        for task_payload in task_payloads:
            ok = mqtt.publish_json_checked(client, task_state_topic, task_payload, qos=1)
            if ok:
                published_task_states.append(task_payload)

mqtt_client.on_message = _on_message
mqtt_client.subscribe(spectator_topic, qos=1)
mqtt_client.subscribe(movement_topic, qos=1)
print('Subscriptions started. Waiting for spectator and movement events...')

Subscription started. Waiting for incoming spectator events...


In [ ]:
# Cell 5 purpose: Run short processing loop, print publish summary, and disconnect safely.
run_for_s = 30
start = time.time()
while time.time() - start < run_for_s:
    time.sleep(0.5)

print(f'Received spectator events: {len(received_spectator_events)}')
print(f'Received movement events: {len(received_movement_events)}')
print(f'Published queue states: {len(published_queue_states)}')
print(f'Published task states: {len(published_task_states)}')

if published_queue_states:
    last_queue = published_queue_states[-1]
    print('Last queue-state payload:')
    print(last_queue)

if published_task_states:
    last_task = published_task_states[-1]
    print('Last task-state payload:')
    print(last_task)

connector = getattr(mqtt_client, '_simcity_connector', None)
if connector is not None:
    connector.disconnect()
    print('Disconnected from MQTT broker.')

print('\n=== Phase 4.2 Facility Manager Complete ===')
print('Queue-state and task-state payloads were published with source run_id continuity.')

Disconnected from MQTT broker (reason=Normal disconnection). Reconnecting...


Received spectator events: 181
Published queue states: 181
Last published queue-state payload:
{'schema_version': '1.0', 'run_id': 'a4-run-eb25aa71', 'timestamp_s': 900, 'source_event_timestamp_s': 900, 'queues': {'zone_a': {'toilet': 85, 'cafe': 65}, 'zone_b': {'toilet': 71, 'cafe': 57}, 'shared_mens_urinal': 48}}
Zone usage snapshot (last payload):
  Zone 1 -> toilet=85, cafe=65
  Zone 2 -> toilet=71, cafe=57
  Shared urinal -> 48
Disconnected from MQTT broker.
